# 01 — Eksploracyjna analiza danych (US Accidents)

Zakres wg rubryki projektu końcowego:
1. **Rozkłady cech** • 2. **Braki danych i ich powody** • 3. **Wartości odstające** • 4. **Nowe cechy**

> Dane: [Kaggle — sobhanmoosavi/us-accidents](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents) → rozpakuj CSV do `dane/US_Accidents_March23.csv`.
> Pracujemy na próbce (`data.sample_rows` w `configs/config.yaml`), pełny zbiór to 7,7 mln wierszy.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from us_accidents_ml.config import load_settings
from us_accidents_ml.data import load_raw

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

settings = load_settings()
df = load_raw(settings)
print(df.shape)

In [ ]:
df.head(3).T

In [ ]:
df.dtypes.value_counts(), df.memory_usage(deep=True).sum() / 1e6

## 1. Braki danych — skala i *powody*

Hipotezy do zweryfikowania: braki pogodowe zależą od odległości do stacji meteo (`Airport_Code`, `Weather_Timestamp`);
`End_Lat/End_Lng` uzupełniane tylko dla części źródeł zgłoszeń (`Source`); braki `Precipitation(in)` ≠ brak opadu.

In [ ]:
missing = (
    df.isna().sum().to_frame("n_missing").assign(pct=lambda t: 100 * t.n_missing / len(df))
)
missing[missing.n_missing > 0].sort_values("pct", ascending=False).head(25)

## 2. Rozkład targetu `Severity` — niezbalansowanie klas

Decyzja do podjęcia w W2: wagi klas vs SMOTE vs łączenie klas skrajnych (por. notatka [[Z02 ML zasoby PMML Kaggle]]).

In [ ]:
counts = df[settings.data.target].value_counts().sort_index()
display(counts.to_frame("n").assign(pct=lambda t: (100 * t.n / len(df)).round(2)))
counts.plot(kind="bar", title="Rozkład Severity")
plt.show()

## 3. Wartości odstające

Kandydaci: `Distance(mi)`, `Temperature(F)`, `Wind_Speed(mph)`, `Pressure(in)`, `Visibility(mi)`
oraz czas trwania zdarzenia (`End_Time - Start_Time`).

In [ ]:
num_cols = ["Distance(mi)", "Temperature(F)", "Wind_Speed(mph)", "Pressure(in)", "Visibility(mi)"]
df[num_cols].describe(percentiles=[0.01, 0.5, 0.99]).T

## 4. Cechy czasowe i geograficzne (nowe cechy)

Plan: godzina/dzień tygodnia/miesiąc, pora doby (`Sunrise_Sunset`), czas trwania, stan/hrabstwo,
gęstość wypadków w okolicy. Poniżej starter.

In [ ]:
start = pd.to_datetime(df["Start_Time"], format="mixed", errors="coerce")
features_time = pd.DataFrame(
    {
        "hour": start.dt.hour,
        "dayofweek": start.dt.dayofweek,
        "month": start.dt.month,
    }
)
features_time.hour.value_counts().sort_index().plot(kind="bar", title="Wypadki wg godziny")
plt.show()

## 5. `Description` — zajawka ścieżki tekstowej (W4)

Sprawdzamy, jak wygląda opis i czy jego długość/słownictwo różnicuje `Severity`.

In [ ]:
print(df["Description"].dropna().sample(5, random_state=42).to_list())
df.assign(desc_len=df["Description"].str.len()).boxplot(
    column="desc_len", by=settings.data.target, showfliers=False
)
plt.show()

## Wnioski (uzupełniać w trakcie W1)

- [ ] Braki: które kolumny odrzucamy, które imputujemy (i czym uzasadniamy)
- [ ] Imbalance: strategia (wagi / SMOTE / merge klas)
- [ ] Outliery: clipping / winsoryzacja / pozostawienie
- [ ] Lista cech do W2 + co przenosimy do ETL (Spark) jako krok przygotowania danych